In [ ]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

metrics = [
    "test/recall@1",
    "test/recall@10",
    "test/recall@50",
    "test/triplet_cosine_accuracy",
    "test/avg_neg_sim",
]

api = wandb.Api()
runs = api.runs("Rec2Vec")

cosine_sweep = []
classic_mse_sweep = []
baseline_100000 = {}

for run in runs:
    history = run.history(keys=metrics, pandas=True)

    try:
        if run.group in ["classic-mse-sweep", "cosine_sweep"]:
            history["run_id"] = run.id
            history["run_name"] = run.name
            history["group"] = run.group
            history["training_style"] = run.config["training-style"]
            history["synthetic_multiplier"] = float(
                run.name.split("synthetic_multiplier-")[1]
            )

            if run.group == "classic-mse-sweep":
                classic_mse_sweep.append(history)
            elif run.group == "cosine_sweep":
                cosine_sweep.append(history)
    except Exception:
        pass

    if "baseline" in run.name and "100000" in run.name:
        baseline_100000 = {metric: history[metric][0] for metric in metrics}

cosine_sweep = pd.concat(cosine_sweep, ignore_index=True)
cosine_sweep.to_csv("cosine_sweep.csv", index=False)

classic_mse_sweep = pd.concat(classic_mse_sweep, ignore_index=True)
classic_mse_sweep.to_csv("classic_mse_sweep.csv", index=False)

df = pd.concat(
    [
        pd.read_csv("cosine_sweep.csv"),
        pd.read_csv("classic_mse_sweep.csv"),
    ],
    ignore_index=True,
)

df["training_style"] = df["training_style"].str.replace(
    "ours-mse",
    "marginal-mse",
)

for metric in metrics:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        data=df,
        x="synthetic_multiplier",
        y=metric,
        hue="training_style",
        marker="o",
        hue_order=["classic-mse", "marginal-mse"],
    )

    if metric in baseline_100000:
        plt.axhline(
            y=baseline_100000[metric],
            color="r",
            linestyle="--",
            label="Baseline",
        )

    plt.xlabel("synthetic_multiplier")
    plt.ylabel(metric)
    plt.title(f"Test {metric} vs Synthetic Multiplier")
    plt.grid(True)
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import wandb
from dateutil import parser


def extract_dataset_size(run_name):
    for part in (run_name or "").split("_"):
        if part.isdigit():
            return part
    return None


def extract_training_style(run_name):
    return (run_name or "").split("_")[0] if run_name else None


def created_ts(run):
    try:
        return parser.parse(run.created_at).timestamp()
    except Exception:
        return 0


def choose_best_baseline(runs):
    if not runs:
        return None
    return sorted(runs, key=lambda r: (len(r.name or ""), -created_ts(r)))[0]


def fetch_run_metrics(run, metrics):
    try:
        history = run.history(keys=metrics, pandas=True)
    except Exception:
        history = pd.DataFrame()

    out = {}
    for metric in metrics:
        val = None

        try:
            val = run.summary.get(metric, None)
        except Exception:
            pass

        if val is None and metric in history.columns:
            vals = history[metric].dropna()
            if len(vals) > 0:
                val = vals.iloc[-1]

        out[metric] = val

    return out


def safe_sort_vals(vals):
    def sort_key(x):
        if x is None:
            return (2, "")
        if isinstance(x, (int, float)):
            return (0, x)
        try:
            return (0, float(x))
        except Exception:
            return (1, str(x))

    return sorted(list(vals), key=sort_key)


def plot_sweep_grid_rows_by_v(
    project_path,
    group_names,
    metrics,
    x_col="easy-negative-value",
    row_col="V",
    training_styles=None,
    figsize_per_plot=(5, 3.5),
    add_baseline=True,
    debug=False,
):
    api = wandb.Api()
    eval_group = "test" if "test" in metrics[0] else "val"

    neg_metric_group = [
        f"{eval_group}/avg_neg_sim",
        f"{eval_group}/avg_easy_neg_sim",
        f"{eval_group}/avg_hard_neg_sim",
    ]
    triplet_metric_group = [
        f"{eval_group}/triplet_cosine_accuracy",
        f"{eval_group}/triplet_accuracy_easy",
        f"{eval_group}/triplet_accuracy_hard",
    ]

    all_metrics_to_fetch = list(set(metrics + neg_metric_group + triplet_metric_group))

    group_runs = list(api.runs(project_path, filters={"group": {"$in": group_names}}))
    if not group_runs:
        print(f"No runs found for groups {group_names}")
        return

    rows = []
    dataset_sizes = set()

    for run in group_runs:
        run_name = run.name or ""
        training_style = run.config.get("training-style") or extract_training_style(run_name)
        dataset_size = extract_dataset_size(run_name)

        if dataset_size is not None:
            dataset_sizes.add(dataset_size)

        row = {
            "run_name": run_name,
            "group": run.group,
            "training_style": training_style,
            "dataset_size": dataset_size,
            x_col: run.config.get(x_col),
            row_col: run.config.get(row_col),
        }
        row.update(fetch_run_metrics(run, all_metrics_to_fetch))
        rows.append(row)

    df = pd.DataFrame(rows)

    if debug:
        print("df shape:", df.shape)
        print("columns:", df.columns.tolist())
        check_cols = [x_col, row_col, "training_style"] + all_metrics_to_fetch
        print(df[[c for c in check_cols if c in df.columns]].head(10))

    needed_cols = ["training_style", "dataset_size", x_col, row_col] + all_metrics_to_fetch
    for col in needed_cols:
        if col not in df.columns:
            df[col] = None

    df = df[df["training_style"].notna() & df[x_col].notna() & df[row_col].notna()].copy()
    if df.empty:
        print("No valid rows to plot.")
        return

    if training_styles is None:
        training_styles = safe_sort_vals(df["training_style"].dropna().unique())

    row_values = safe_sort_vals(df[row_col].dropna().unique())

    baselines_by_size = {}
    if add_baseline:
        all_runs = list(api.runs(project_path))
        baseline_runs = [r for r in all_runs if "baseline" in (r.name or "").lower()]

        for dataset_size in dataset_sizes:
            candidates = [r for r in baseline_runs if dataset_size in (r.name or "")]
            best = choose_best_baseline(candidates)

            if best:
                baselines_by_size[dataset_size] = fetch_run_metrics(best, all_metrics_to_fetch)
                print(f"Baseline ({dataset_size}): {best.name}")
            else:
                print(f"No baseline for {dataset_size}")

    fig, axes = plt.subplots(
        len(row_values),
        len(metrics),
        figsize=(figsize_per_plot[0] * len(metrics), figsize_per_plot[1] * len(row_values)),
        squeeze=False,
    )

    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    color_map = {style: colors[i % len(colors)] for i, style in enumerate(training_styles)}
    linestyle_map = {"overall": "-", "easy": ":", "hard": "--"}

    for i, row_val in enumerate(row_values):
        row_df = df[df[row_col] == row_val].copy()

        for j, metric in enumerate(metrics):
            ax = axes[i][j]
            plotted = False

            if metric == f"{eval_group}/avg_neg_sim":
                grouped_variants = [
                    (f"{eval_group}/avg_neg_sim", "overall"),
                    (f"{eval_group}/avg_easy_neg_sim", "easy"),
                    (f"{eval_group}/avg_hard_neg_sim", "hard"),
                ]
            elif metric == f"{eval_group}/triplet_cosine_accuracy":
                grouped_variants = [
                    (f"{eval_group}/triplet_cosine_accuracy", "overall"),
                    (f"{eval_group}/triplet_accuracy_easy", "easy"),
                    (f"{eval_group}/triplet_accuracy_hard", "hard"),
                ]
            else:
                grouped_variants = [(metric, "overall")]

            for style in training_styles:
                style_df = row_df[row_df["training_style"] == style].sort_values(by=x_col)

                for variant, kind in grouped_variants:
                    if variant not in style_df.columns:
                        continue

                    sub = style_df[[x_col, variant]].dropna()
                    if sub.empty:
                        continue

                    ax.plot(
                        sub[x_col],
                        sub[variant],
                        marker="o",
                        linestyle=linestyle_map.get(kind, "-"),
                        color=color_map[style],
                    )
                    plotted = True

            if add_baseline:
                for ds in safe_sort_vals(row_df["dataset_size"].dropna().unique()):
                    baseline_metrics = baselines_by_size.get(ds, {})
                    for variant, kind in grouped_variants:
                        baseline_val = baseline_metrics.get(variant)
                        if baseline_val is None:
                            continue

                        ax.axhline(
                            y=baseline_val,
                            linestyle=linestyle_map.get(kind, "dashdot"),
                            color="red",
                            alpha=0.8,
                        )
                        plotted = True

            if plotted:
                if i == 0:
                    ax.set_title(metric)
                ax.set_ylabel(f"{row_col}={row_val}" if j == 0 else "")
                ax.set_xlabel(x_col)
                ax.grid(True)
            else:
                ax.set_visible(False)

    handles = []
    labels = []

    for style in training_styles:
        handles.append(plt.Line2D([0], [0], color=color_map[style], linestyle="-", marker="o"))
        labels.append(style)

    if f"{eval_group}/avg_neg_sim" in metrics or f"{eval_group}/triplet_cosine_accuracy" in metrics:
        for label, linestyle in [("overall", "-"), ("easy", ":"), ("hard", "--")]:
            handles.append(plt.Line2D([0], [0], color="black", linestyle=linestyle))
            labels.append(label)

    if add_baseline:
        handles.append(plt.Line2D([0], [0], color="red", linestyle="dashdot"))
        labels.append("baseline")

    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=min(len(labels), 6),
        fontsize=9,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


In [1]:
api = wandb.Api()

runs = api.runs(
    "Rec2Vec",
    filters={"group": "easy-neg-sweep2"},
)

all_data = []

for run in runs:
    history = run.history(keys=metrics, pandas=True)

    try:
        if run.group == "easy-neg-sweep2":
            history["run_id"] = run.id
            history["run_name"] = run.name
            history["training_style"] = run.config["training-style"]
            history["easy_neg_value"] = float(run.name.split("easy-")[1])
            all_data.append(history)
    except Exception:
        pass

easy_neg_sweep = pd.concat(all_data, ignore_index=True)
easy_neg_sweep.to_csv("easy_neg_sweep.csv", index=False)

for metric in metrics:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        data=easy_neg_sweep,
        x="easy_neg_value",
        y=metric,
        hue="training_style",
        marker="o",
    )

    if metric in baseline_100000:
        plt.axhline(
            y=baseline_100000[metric],
            color="r",
            linestyle="--",
            label="Baseline",
        )

    plt.xlabel("easy negative value")
    plt.ylabel(metric)
    plt.title(f"Test {metric} vs Easy Negative Value")
    plt.grid(True)
    plt.show()


NameError: name 'wandb' is not defined

In [ ]:
metrics = [
    "val/recall@1",
    "val/recall@10",
    "val/recall@50",
    "val/triplet_cosine_accuracy",
    "val/avg_neg_sim",
]

plot_sweep_grid_rows_by_v(
    project_path="Rec2Vec",
    group_names=["sweep_val"],
    metrics=metrics,
    x_col="easy-negative-value",
    row_col="V",
    debug=False,
)
# 274812 train, 34348 eval, 34354 test examples


In [ ]:
metrics = [
    "val/recall@1",
    "val/recall@10",
    "val/recall@50",
    "val/triplet_cosine_accuracy",
    "val/avg_neg_sim",
]

plot_sweep_grid_rows_by_v(
    project_path="Rec2Vec",
    group_names=["rephrased_sweep"],
    metrics=metrics,
    x_col="easy-negative-value",
    row_col="V",
    debug=False,
)
# 274812 train, 34348 eval, 34354 test examples


In [ ]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

api = wandb.Api()

entity = "sc5780-columbia-university"
project = "Rec2Vec"
run_ids = ["fqr0kk9v", "zg0uztkx", "ss2tlxnb", "7pser4fv"]

metrics = [
    "val/recall@1",
    "val/recall@10",
    "val/recall@50",
]

rows = []
for run_id in run_ids:
    run = api.run(f"{entity}/{project}/{run_id}")
    summary = run.summary

    model_name = (
        summary.get("model_name")
        or run.config.get("model_name")
        or run.name
    )
    model_label = model_name.split("/")[-1]

    row = {
        "run_id": run_id,
        "model": model_label,
        "run_name": run.name,
    }
    for metric in metrics:
        row[metric] = summary.get(metric, np.nan)
    rows.append(row)

df = pd.DataFrame(rows)
display(df)

plot_df = df.set_index("model")[metrics]
ax = plot_df.plot(kind="bar", figsize=(10, 5), width=0.8)

ax.set_xlabel("Model")
ax.set_ylabel("Metric value")
ax.set_title("Model Comparison on Recall")
ax.legend(title="Metric")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


In [1]:
import ast
import json
import re
from pathlib import Path

import pandas as pd
import yaml

RUNS = [
    ("Text", "Triplet", "Text", "7sucoix6"),
    ("Text", "Classic MSE", "Text", "rpzlgy40"),
    ("Text", "Marginal MSE (ours)", "Text", "hj8jjy3v"),
    ("Text+Image", "Triplet", "Image", "mjpnf0vo"),
    ("Text+Image", "Classic MSE", "Image", "j74gj26i"),
    ("Text+Image", "Marginal MSE (ours)", "Image", "0dl9yyb1"),
    ("Image", "Triplet", "Image", "bg5thphv"),
    ("Image", "Classic MSE", "Image", "c0d1918l"),  # likely unfinished
    ("Image", "Marginal MSE (ours)", "Image", "hybnkqzb"),
]

WANDB_DIR = Path("wandb")


def find_run_dir(run_id):
    matches = sorted(WANDB_DIR.glob(f"run-*-{run_id}"))
    return matches[-1] if matches else None


def get_args(run_dir):
    config_path = run_dir / "files" / "config.yaml"
    if not config_path.exists():
        return ""
    config = yaml.safe_load(config_path.read_text())
    try:
        event = next(iter(config["_wandb"]["value"]["e"].values()))
        return " ".join(map(str, event.get("args", [])))
    except Exception:
        return ""


def read_summary(run_dir):
    path = run_dir / "files" / "wandb-summary.json"
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text())
    except Exception:
        return {}


def read_output_dicts(run_dir):
    path = run_dir / "files" / "output.log"
    if not path.exists():
        return []

    text = path.read_text(errors="ignore")
    dicts = []
    for match in re.finditer(r"\{[^\n{}]*(?:recall|accuracy)@5[^\n{}]*\}", text):
        try:
            dicts.append(ast.literal_eval(match.group(0)))
        except Exception:
            pass
    return dicts


def pick_metric(metrics, k):
    candidates = [
        f"val/recall@{k}",
        f"test/recall@{k}",
        f"val/val_recall@{k}",
        f"test/test_recall@{k}",
        f"val_cosine_recall@{k}",
        f"test_cosine_recall@{k}",
        f"cosine_recall@{k}",
        f"val_recall@{k}",
        f"test_recall@{k}",
        f"recall@{k}",
        f"val_cosine_accuracy@{k}",
        f"test_cosine_accuracy@{k}",
        f"cosine_accuracy@{k}",
        f"val_accuracy@{k}",
        f"test_accuracy@{k}",
        f"accuracy@{k}",
    ]
    for key in candidates:
        if key in metrics:
            return metrics[key]
    return None


def extract_recalls(run_dir):
    summary = read_summary(run_dir)

    merged = dict(summary)

    # Fallback to evaluator dicts printed in output.log. Later dicts win.
    for d in read_output_dicts(run_dir):
        merged.update(d)

    return {
        "recall@5": pick_metric(merged, 5),
        "recall@10": pick_metric(merged, 10),
        "recall@100": pick_metric(merged, 100),
    }


rows = []
for trained_on, method, evaluated_on, run_id in RUNS:
    run_dir = find_run_dir(run_id)
    args = get_args(run_dir) if run_dir else ""
    recalls = extract_recalls(run_dir) if run_dir else {"recall@5": None, "recall@10": None, "recall@100": None}

    rows.append({
        "trained on": trained_on,
        "method": method,
        "Evaluated on": evaluated_on,
        "run_id": run_id,
        "run_dir": str(run_dir) if run_dir else None,
        "recall@5": recalls["recall@5"],
        "recall@10": recalls["recall@10"],
        "recall@100": recalls["recall@100"],
        "verified_args": args,
    })

df = pd.DataFrame(rows)
display(df[["trained on", "method", "Evaluated on", "run_id", "recall@5", "recall@10", "recall@100"]])

# Optional: inspect command-line args used for each run.
display(df[["run_id", "run_dir", "verified_args"]])


,trained on,method,Evaluated on,run_id,recall@5,recall@10,recall@100
0,Text,Triplet,Text,7sucoix6,0.724752,0.789556,0.929393
1,Text,Classic MSE,Text,rpzlgy40,0.724260,0.800669,0.957911
2,Text,Marginal MSE (ours),Text,hj8jjy3v,0.780509,0.845708,0.963222
3,Text+Image,Triplet,Image,mjpnf0vo,0.146605,0.212963,0.553241
4,Text+Image,Classic MSE,Image,j74gj26i,0.128858,0.203704,0.550154
5,Text+Image,Marginal MSE (ours),Image,0dl9yyb1,0.177469,0.263889,0.625772
6,Image,Triplet,Image,bg5thphv,0.163580,0.226080,0.592593
7,Image,Classic MSE,Image,c0d1918l,0.162809,0.229938,0.598765
8,Image,Marginal MSE (ours),Image,hybnkqzb,0.195988,0.283179,0.647377


,run_id,run_dir,verified_args
0,7sucoix6,wandb/run-20260502_052426-7sucoix6,--V=20 --dataset=dataset/processed/feature-dis...
1,rpzlgy40,wandb/run-20260501_135213-rpzlgy40,--V=20 --dataset=dataset/processed/feature-dis...
2,hj8jjy3v,wandb/run-20260501_132029-hj8jjy3v,--V=20 --dataset=dataset/processed/feature-dis...
3,mjpnf0vo,wandb/run-20260503_155821-mjpnf0vo,--dataset dataset/processed/deepfashion-inshop...
4,j74gj26i,wandb/run-20260505_163130-j74gj26i,--dataset dataset/processed/deepfashion-inshop...
5,0dl9yyb1,wandb/run-20260503_173924-0dl9yyb1,--dataset dataset/processed/deepfashion-inshop...
6,bg5thphv,wandb/run-20260510_190212-bg5thphv,--dataset dataset/processed/deepfashion-inshop...
7,c0d1918l,wandb/run-20260510_190444-c0d1918l,--dataset dataset/processed/deepfashion-inshop...
8,hybnkqzb,wandb/run-20260510_190327-hybnkqzb,--dataset dataset/processed/deepfashion-inshop...
